**Question 1. Install Spark and PySpark**

**Answer:**
Installed Java 17+, then installed PySpark using uv in a project environment:
```bash
uv init
uv add pyspark
```
Verified installation by creating a SparkSession and running a simple Spark job.

In [4]:
import pyspark
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/09 11:13:16 WARN Utils: Your hostname, Nikhil, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/09 11:13:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 11:13:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [21]:
spark.version

'4.1.1'

**Question 2. Yellow November 2025**

In [9]:
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")
df.repartition(4).write.parquet("hw_data/")

In [11]:
df.printSchema()


root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



**Question 3. Count records**

In [14]:
from pyspark.sql import functions as F
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")
df \
    .withColumn('pickup_date', F.to_date(df.tpep_pickup_datetime)) \
    .filter("pickup_date = '2025-11-15'") \
    .count()

162604

**Question 4. Longest trip**

In [15]:
from pyspark.sql import functions as F

df = spark.read.parquet("yellow_tripdata_2025-11.parquet")

df.select(
    (F.max(F.unix_timestamp("tpep_dropoff_datetime") -
           F.unix_timestamp("tpep_pickup_datetime")) / 3600)
    .alias("longest_trip_hours")
).show()

+------------------+
|longest_trip_hours|
+------------------+
| 90.64666666666666|
+------------------+



**Question 5. User Interface**

**Answer:**
✅4040

**Question 6. Least frequent pickup location zone**

In [19]:
trips = spark.read.parquet("yellow_tripdata_2025-11.parquet")
zones = spark.read.option("header", True).csv("taxi_zone_lookup.csv")

result = (
    trips
    .groupBy("PULocationID")
    .count()
    .join(zones, trips.PULocationID == zones.LocationID)
    .orderBy("count")
)

result.select("Zone","count").show(10)

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|Governor's Island...|    1|
|       Arden Heights|    1|
|Eltingville/Annad...|    1|
|       Port Richmond|    3|
|   Rossville/Woodrow|    4|
|       Rikers Island|    4|
| Green-Wood Cemetery|    4|
|         Great Kills|    4|
|         Jamaica Bay|    5|
|         Westerleigh|   12|
+--------------------+-----+
only showing top 10 rows


In [20]:
spark.stop()